In [1]:
from pathlib import Path

PAPER_PATH = Path("../papers/")
TEST_PAPERS = [
#    PAPER_PATH / "2607.00283v1.pdf",
    PAPER_PATH / "2607.00292v1.pdf",
]
PAGES_TO_TEST = 9  # keep this small for a quick eval

## PyMuPDF text extraction vs. Qwen3-VL-8B-Instruct vs. LlamaParse

There are many PDF parsing solutions, and we want to test which solution will best fit are needs. We need a solution that will balance speed and accuracy, and the solution must achieve reasonable performance on extracting text on tables. However, we are willing to sacrifice performance on complex figures and mathematical equations.

We will compare the following methods in this notebook:

1. **PyMuPDF baseline** — native text-layer extraction
2. **Qwen3-VL** — local vision-language model reading rendered page images
3. **LlamaParse (llama_cloud)** — hosted parsing API, `tier="agentic_plus"`


In [2]:
import os

# Set BEFORE importing transformers/torch so the cache lands on E: instead of C:
os.environ.setdefault("HF_HOME", "E:/hf_cache")

import pymupdf4llm
from PIL import Image
from dotenv import load_dotenv

load_dotenv()

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

## Defining Functions for Processing PDF as Image

We define the following functions for processing the PDFs
- `render_pages:` Takes in a .pdf file path and converts it to a list of PIL Images
- `baseline_text:` Takes in a .pdf file path and runs it through PyMuPDF (baseline) solution

In [3]:
def render_pages(pdf_path: Path, n_pages: int, dpi: int = 200) -> list[Image.Image]:
    doc = pymupdf4llm.pymupdf.open(pdf_path)
    images = []
    for i in range(min(n_pages, doc.page_count)):
        pix = doc[i].get_pixmap(dpi=dpi)
        images.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
    doc.close()
    return images


def baseline_text(pdf_path: Path, n_pages: int) -> list[str]:
    doc = pymupdf4llm.pymupdf.open(pdf_path)
    n = min(n_pages, doc.page_count)
    chunks = pymupdf4llm.to_markdown(
        doc, pages=list(range(n)), page_chunks=True, use_ocr=True
    )
    doc.close()
    return [chunk["text"] for chunk in chunks]

## Load Local Qwen Model into Memory

Now, we load Qwen3-VL  and its processor using 4-bit quantization to keep memory usage low on local GPU hardware.

We've also added a fallback in case bitsandbytes isn't available on our Python version.

> We were unable to successful run the Qwen model locally as our hardware was limited to 6GB of VRAM and 32 GB of RAM.
> Thus, Qwen 3 results are empty for the rest of the analysis


In [4]:
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

try:
    from transformers import BitsAndBytesConfig

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        llm_int8_enable_fp32_cpu_offload=True,
    )
except ImportError:
    # bitsandbytes not available for this Python version -> fall back to
    # unquantized bf16, letting accelerate split layers across GPU/CPU.
    print("Quantization failed!")
    quant_config = None

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
    quantization_config=quant_config,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

e:\engineer\paper-iq\.venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0811 11:43:34.225000 22020 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 625/625 [00:10<00:00, 59.53it/s] 


[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in e:\engineer\paper-iq\.venv313\Lib\site-packages\transformers\models\qwen3_vl\video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in e:\engineer\paper-iq\.venv313\Lib\site-packages\transformers\models\qwen3_vl\video_processing_qwen3_vl.py.


## Define Prompt Structure for VLM

Here, we define a prompt for the VLM to parse the .pdf files.

In [5]:
PROMPT = (
    "Transcribe this document page to plain markdown, preserving reading order. "
    "Represent tables as markdown tables and equations as LaTeX."
)


def vlm_extract(image: Image.Image, max_new_tokens: int = 1024) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    generated = model.generate(**inputs, max_new_tokens=max_new_tokens)
    trimmed = generated[0][inputs["input_ids"].shape[1] :]
    return processor.decode(trimmed, skip_special_tokens=True)

### LlamaParse (llama_cloud) setup

Here we define a function for llama cloud to parse PDF files


In [6]:
from llama_cloud import LlamaCloud

# LlamaCloud() reads LLAMA_CLOUD_API_KEY from the environment -- already loaded
# by the load_dotenv() call earlier in this notebook.
llama_client = LlamaCloud()


def llama_extract(pdf_path: Path, n_pages: int) -> list[str]:
    with open(pdf_path, "rb") as f:
        file_obj = llama_client.files.create(file=f, purpose="parse")

    result = llama_client.parsing.parse(
        file_id=file_obj.id,
        tier="agentic_plus",
        version="latest",
        expand=["markdown"],
        page_ranges={"max_pages": n_pages},
    )

    pages_by_number = {p.page_number: p for p in result.markdown.pages}
    texts = []
    for page_number in range(1, n_pages + 1):
        page = pages_by_number.get(page_number)
        if page is not None and page.success:
            texts.append(page.markdown)
        else:
            texts.append("")
    return texts

## Benchmarking All Solutions: PyMuPDF4LLM, LlamaParse, VLM extraction

This loop times and collects PDF-to-markdown output across three methods, per test paper, so they can be compared on both speed and quality.

> Again, note that the 


In [7]:
import time

results = []

for paper in TEST_PAPERS:
    pages = render_pages(paper, PAGES_TO_TEST)

    start_baseline = time.perf_counter()
    baseline_pages = baseline_text(paper, PAGES_TO_TEST)
    baseline_time = time.perf_counter() - start_baseline
    print("baseline finished!")

    start_llama = time.perf_counter()
    llama_pages = llama_extract(paper, PAGES_TO_TEST)
    llama_time = time.perf_counter() - start_llama
    print("llama_cloud finished!")

    for i, (img, baseline, llama_out) in enumerate(zip(pages, baseline_pages, llama_pages)):

        start_vlm = time.perf_counter()
        vlm_out = ""# vlm_extract(img)
        vlm_time = time.perf_counter() - start_vlm

        results.append(
            {
                "paper": paper.name,
                "page": i,
                "baseline": baseline,
                "llama": llama_out,
                "vlm": vlm_out,
                "baseline_time": baseline_time,
                "llama_time": llama_time,
                "vlm_time": vlm_time,
            }
        )
        print("page processed!")


=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.
baseline finished!
llama_cloud finished!
page processed!
page processed!
page processed!
page processed!
page processed!
page processed!
page processed!
page processed!
page processed!


In [14]:
results

[{'paper': '2607.00292v1.pdf',
  'page': 0,
  'baseline': '# An LLM-Based Framework for Intent-Driven Network Topology Design \n\nKholoud El Habbouli, Fen Zhou and Stéphane Huet \n\n_† CERI-LIA, University of Avignon, France_ Emails : {firstname.lastname@univ-avignon.fr} \n\n**_Abstract_ —Designing deployable and resilient network topologies from natural language requirements remains a challenging problem in network automation. This work investigates the ability of Large Language Models (LLMs) to generate structurally valid and constraint-compliant network topologies through a constraint-driven pipeline combining hierarchical modeling and systematic validation. The framework is evaluated via a multimodel comparison of proprietary and open-weight LLMs across four realistic network scenarios released as a public dataset. We assess structural correctness using node and edge F1-scores against reference topologies, and evaluate resilience through server and content connectivity metrics. In 

In [8]:
# Eyeball comparison -- there's no ground-truth transcript, so judge each pair on:
# completeness, table/equation fidelity, and reading order.
for r in results:
    print(f"===== {r['paper']} — page {r['page']} =====")
    print("--- PyMuPDF baseline ---")
    print(r["baseline"][:10])
    print("\n--- LlamaParse ---")
    print(r["llama"][:10])
    print("\n--- Qwen3-VL ---")
    print(r["vlm"][:10])
    print()

===== 2607.00292v1.pdf — page 0 =====
--- PyMuPDF baseline ---
# An LLM-B

--- LlamaParse ---
# An LLM-B

--- Qwen3-VL ---


===== 2607.00292v1.pdf — page 1 =====
--- PyMuPDF baseline ---
## based r

--- LlamaParse ---
based resi

--- Qwen3-VL ---


===== 2607.00292v1.pdf — page 2 =====
--- PyMuPDF baseline ---


<!-- Sta

--- LlamaParse ---
<table>
  

--- Qwen3-VL ---


===== 2607.00292v1.pdf — page 3 =====
--- PyMuPDF baseline ---
cable. Alt

--- LlamaParse ---
<table>
  

--- Qwen3-VL ---


===== 2607.00292v1.pdf — page 4 =====
--- PyMuPDF baseline ---
violations

--- LlamaParse ---
violations

--- Qwen3-VL ---


===== 2607.00292v1.pdf — page 5 =====
--- PyMuPDF baseline ---
as missing

--- LlamaParse ---
TABLE I: D

--- Qwen3-VL ---


===== 2607.00292v1.pdf — page 6 =====
--- PyMuPDF baseline ---
TABLE II: 

--- LlamaParse ---
TABLE II: 

--- Qwen3-VL ---


===== 2607.00292v1.pdf — page 7 =====
--- PyMuPDF baseline ---






<!--

--- LlamaParse ---
TABLE III:

--- Qwen3-VL ---




## LLM-judged fidelity grading

Use a Claude model as a judge: show it the rendered page image (ground truth) plus both
extractions, and ask it to surface concrete issues and grade each on fidelity.

Setup:

```bash
pip install anthropic
```

Add the anthropic key to `.env` (same file as `HF_TOKEN`):

```
ANTHROPIC_API_KEY=sk-ant-...
```


In [9]:
import base64
import json
from io import BytesIO

from anthropic import Anthropic

# Anthropic() reads ANTHROPIC_API_KEY from the environment -- already loaded by
# the load_dotenv() call earlier in this notebook.
client = Anthropic()
JUDGE_MODEL = "claude-sonnet-5"

JUDGE_PROMPT = """You are grading two automated text extractions of the PDF page shown \
in the attached image.

Method A ("baseline") is native PDF text-layer extraction (no OCR, no model).
Method B ("llama") is LlamaParse (hosted parsing API) transcription of the same page.

Using the image as ground truth, identify concrete issues with each extraction: \
missing content, garbled or missing tables/equations, wrong reading order, \
hallucinated text, formatting problems. Then grade each on fidelity to the page, \
from 1 (unusable) to 10 (perfect, complete, correctly ordered).

--- Method A (baseline) ---
{baseline}

--- Method C (llama) ---
{llama}

Respond with ONLY a JSON object, no surrounding prose or markdown fences, in this \
exact shape:
{{"baseline_grade": <int 1-10>, "baseline_issues": [<str>, ...], \
"vlm_grade": <int 1-10>, "vlm_issues": [<str>, ...], \
"llama_grade": <int 1-10>, "llama_issues": [<str>, ...]}}"""


def image_to_base64(image: Image.Image) -> str:
    buf = BytesIO()
    image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def judge_page(image: Image.Image, baseline: str, llama: str) -> dict:
    response = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=1536,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": image_to_base64(image),
                        },
                    },
                    {
                        "type": "text",
                        "text": JUDGE_PROMPT.format(
                            baseline=baseline, llama=llama
                        ),
                    },
                ],
            }
        ],
    )
    # claude-sonnet-5 runs adaptive thinking by default, so content[0] may be
    # a ThinkingBlock rather than the TextBlock -- find the text block instead
    # of indexing.
    raw = next(block.text for block in response.content if block.type == "text").strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)

In [10]:
# Re-render pages so the judge sees the same image each method was tested against
# (cheap: no model call, just PDF -> image, cached per paper).
page_images = {paper.name: render_pages(paper, PAGES_TO_TEST) for paper in TEST_PAPERS}

judged = []
for r in results:
    img = page_images[r["paper"]][r["page"]]
    verdict = judge_page(img, r["baseline"], r["llama"])
    judged.append({**r, **verdict})
    print(
        f"{r['paper']} page {r['page']}: "
        f"baseline={verdict['baseline_grade']}/10, "
        f"llama={verdict['llama_grade']}/10"
    )

2607.00292v1.pdf page 0: baseline=9/10, llama=9/10
2607.00292v1.pdf page 1: baseline=10/10, llama=10/10
2607.00292v1.pdf page 2: baseline=8/10, llama=3/10
2607.00292v1.pdf page 3: baseline=8/10, llama=3/10
2607.00292v1.pdf page 4: baseline=8/10, llama=10/10
2607.00292v1.pdf page 5: baseline=9/10, llama=4/10
2607.00292v1.pdf page 6: baseline=9/10, llama=5/10
2607.00292v1.pdf page 7: baseline=9/10, llama=3/10
2607.00292v1.pdf page 8: baseline=4/10, llama=9/10


In [11]:
for j in judged:
    print(f"===== {j['paper']} — page {j['page']} =====")
    print(f"baseline: {j['baseline_grade']}/10")
    for issue in j["baseline_issues"]:
        print(f"  - {issue}")
    print(f"llama: {j['llama_grade']}/10")
    for issue in j["llama_issues"]:
        print(f"  - {issue}")
    print()

===== 2607.00292v1.pdf — page 0 =====
baseline: 9/10
  - Minor markdown formatting artifacts (underscores for italics)
  - Otherwise complete and accurate extraction
llama: 9/10
  - Minor extra spacing before 'such as NetConfEval' paragraph
  - Otherwise complete and accurate extraction

===== 2607.00292v1.pdf — page 1 =====
baseline: 10/10
llama: 10/10

===== 2607.00292v1.pdf — page 2 =====
baseline: 8/10
  - Diagram text extracted as garbled meaningless characters (i, 6, 2B) instead of being omitted or described
  - Minor formatting artifact with superscript notation in int_x^vi
  - Otherwise correct reading order and complete body text
llama: 3/10
  - Reading order severely scrambled - paragraphs repeated and out of sequence
  - Duplicated text blocks (e.g., 'From this use case' paragraph appears twice, 'and CC objectives' text repeated)
  - Invented fabricated tables not present in source (Network Topology Components, Node ID connectivity table) - hallucinated content
  - Fabricate

## Findings

**baseline (pymupdf4llm)**
- Overall more consistent results -- average score is 8.22/10
- Fails gracefully on equations by leaving blank or omitting altogether
- Multi-column reading order preserved correctly in almost all cases
- Main weakness is complex tables (e.g. Table IV on page 8, scored 4/10) where rows/columns get misaligned during markdown conversion
- Fast and cheap, making it viable as the default parser at scale

**LlamaParse**
- Average score of 6.22/10
- Struggles with hallucinated tables, reading order, and section headers
- When it works (single-column text pages, e.g. pages 1 and 4), it outperforms baseline (10/10 vs 8-10/10) and produces cleaner HTML tables (page 8: 9/10 vs baseline's 4/10)

**Decision:** use pymupdf4llm as the default parser.

## Run PyMuPDF4LLM on Sample

We want to run the chosen solution on a sample of ~100 papers to choose a chunking approach.

Consider the following approaches:
- **Fixed-size chunking:** This solution is relatively easy to implement however we want to be careful of size we choose. We plan on using this as our baseline.
- **Structure-aware chunking:** We will use section headers to delimit chunks, so each chunk maps to a logical unit of the paper (e.g. Introduction, Method, Results) rather than an arbitrary token boundary.
- **Hybrid section + paragraph chunking:** We chunk by section first, then recursively split any section that's still too large by paragraph -- this should give us structure-aware chunks that also stay within a target size.

In [22]:
import numpy as np
results_sample = []

test_papers = list(PAPER_PATH.glob('*.pdf'))
sample_idx = np.random.choice(len(test_papers),
                              size=100,
                              replace=False)
test_papers_sample = [test_papers[i] for i in sample_idx]

for paper in test_papers_sample:

    start_baseline = time.perf_counter()
    baseline_pages = baseline_text(paper, PAGES_TO_TEST)
    baseline_time = time.perf_counter() - start_baseline
    print("baseline finished!")

    for i, baseline in enumerate(baseline_pages):
        results_sample.append(
            {
                "paper": paper.name,
                "page": i,
                "baseline": baseline,
                "baseline_time": baseline_time,
            }
            )
        print(f"Processed page {i}")


=== Document parser messages ===
Using Tesseract for OCR processing.
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=3/4.
OCR on page.number=7/8.
baseline finished!
Processed page 0
Processed page 1
Processed page 2
Processed page 3
Processed page 4
Processed page 5
Processed page 6
Processed page 7
Processed page 8

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.
processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=3/4.
OCR on page.number=7/8.
baseline finished!
Processed page 0
Processed page 1
Processed page 2
Processed page 3
Processed page 4
Processed page 5
Processed page 6
Processed page 7
Processed page 8

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=5/6.
OCR on page.number=6/7.
baseline finished!
Processed page 0
Processed page 1
Processed

e:\engineer\paper-iq\.venv313\Lib\site-packages\pymupdf4llm\ocr\compute_ocr_features.py:236: RuntimeWarning: Mean of empty slice
  fft_ratio = magnitude[magnitude > magnitude.mean()].mean() / (
e:\engineer\paper-iq\.venv313\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=1/2.
baseline finished!
Processed page 0
Processed page 1
Processed page 2
Processed page 3
Processed page 4
Processed page 5
Processed page 6
Processed page 7
Processed page 8

=== Document parser messages ===
Using Tesseract for OCR processing.
baseline finished!
Processed page 0
Processed page 1
Processed page 2
Processed page 3
Processed page 4
Processed page 5
Processed page 6
Processed page 7
Processed page 8

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=4/5.
OCR on page.number=5/6.
baseline finished!
Processed page 0
Processed page 1
Processed page 2
Processed page 3
Processed page 4
Processed page 5
Processed page 6

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=4/5.
OCR on page.number=7/8.
OCR on page.number=5/6.
baseline finished!
Processed page 0
Processed page 1
Processed page 2


In [9]:
import pickle

with open("parsing_results_sample.pickle", "rb") as f:
    results_sample = pickle.load(f)

In [12]:
results_sample[0]

{'paper': '2607.11008v1.pdf',
 'page': 0,
 'baseline': '# **SynCLIP: Synonym-Coherent Language-Image Pretraining for Robust Open-Vocabulary Dense Perception** \n\nMingjie Xie<sup>1</sup> Guangjun He<sup>2*</sup> Dongli Xu<sup>3</sup> Youtian Lin<sup>4</sup> Hongjue Li<sup>1</sup> Pengming Feng<sup>2</sup> Jian Guan<sup>5</sup> Yue Deng<sup>1</sup><sup>_,_6</sup> \n\n1Beihang University \n\n2State Key Laboratory of Space Information System and Integrated Application \n\n3Independent Researcher 4Nanjing University \n\n5Harbin Engineering University 6Beijing Zhongguancun Academy \n\n## **Abstract** \n\n_Open-vocabulary dense perception (OVDP) aims to localize objects unseen during training by leveraging textual knowledge. Despite the remarkable progress of recent CLIP-based approaches, we identify a critical limitation: synonym-induced grounding inconsistency, where semantically equivalent expressions yield disparate spatial attention patterns. This inconsistency undermines the robustness